# 💼 Gestion et Analyse des Données en Python
**COT_GenAI 2026 — Semaine 2, Jour 2 — Défi Quotidien**

Dataset : **Data Science Job Salaries**

### Objectifs :
1. Normalisation Min-Max de la colonne `salary_in_usd`
2. Réduction de dimensionnalité (PCA + t-SNE)
3. Agrégation par niveau d'expérience (salaire moyen & médian)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import warnings
warnings.filterwarnings('ignore')

# Style des graphiques
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ Imports réussis.')

---
## 📥 Chargement du Dataset
Télécharger le fichier `ds_salaries.csv` depuis [Kaggle](https://www.kaggle.com/datasets/ruchi798/data-science-job-salaries) et le placer dans le même dossier que ce notebook.

In [ ]:
# Chargement du dataset
df = pd.read_csv('ds_salaries.csv')

print(f'Dimensions : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
display(df.head())

In [ ]:
# Exploration rapide
print('=== Informations générales ===')
df.info()

print('\n=== Valeurs manquantes ===')
print(df.isnull().sum())

print('\n=== Statistiques descriptives (colonnes numériques) ===')
display(df.describe())

In [ ]:
# Valeurs uniques des colonnes catégorielles clés
print("Niveaux d'expérience :", df['experience_level'].unique())
print("Types d'emploi :", df['employment_type'].unique())
print("Tailles d'entreprise :", df['company_size'].unique())
print(f"\nDistribution des niveaux d'expérience :")
print(df['experience_level'].value_counts())

---
## 🌟 Tâche 1 : Normalisation Min-Max de la colonne `salary_in_usd`

La normalisation Min-Max transforme toutes les valeurs dans l'intervalle **[0, 1]** :

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Utile car elle empêche les salaires élevés de dominer les algorithmes de ML sensibles à l'échelle (KNN, réseaux de neurones...).

In [ ]:
# Visualisation de la distribution AVANT normalisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['salary_in_usd'].hist(ax=axes[0], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution salary_in_usd (AVANT normalisation)')
axes[0].set_xlabel('Salaire (USD)')
axes[0].set_ylabel('Fréquence')

df.boxplot(column='salary_in_usd', ax=axes[1])
axes[1].set_title('Boxplot salary_in_usd (AVANT normalisation)')

plt.tight_layout()
plt.show()

print(f"Min : {df['salary_in_usd'].min():,.0f} $ | Max : {df['salary_in_usd'].max():,.0f} $ | Moyenne : {df['salary_in_usd'].mean():,.0f} $")

In [ ]:
# Application du MinMaxScaler
scaler = MinMaxScaler()
df['salary_normalized'] = scaler.fit_transform(df[['salary_in_usd']])

print('✅ Colonne salary_normalized créée.')
print(f"Min normalisé : {df['salary_normalized'].min():.4f}")
print(f"Max normalisé : {df['salary_normalized'].max():.4f}")
print(f"Moyenne normalisée : {df['salary_normalized'].mean():.4f}")

display(df[['salary_in_usd', 'salary_normalized']].head(10))

In [ ]:
# Visualisation AVANT vs APRÈS normalisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['salary_in_usd'].hist(ax=axes[0], bins=40, color='salmon', edgecolor='white')
axes[0].set_title('AVANT normalisation')
axes[0].set_xlabel('Salaire (USD)')

df['salary_normalized'].hist(ax=axes[1], bins=40, color='teal', edgecolor='white')
axes[1].set_title('APRÈS normalisation Min-Max')
axes[1].set_xlabel('Salaire normalisé [0, 1]')

plt.suptitle('Normalisation Min-Max — salary_in_usd', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Note : la forme de la distribution ne change pas, seule l'échelle change.
print('💡 La forme de la distribution est préservée, seule l\'échelle change.')

---
## 🌟 Tâche 2 : Réduction de Dimensionnalité (PCA + t-SNE)

- **PCA** : préserve la variance globale, rapide, interprétable
- **t-SNE** : révèle les clusters et structures locales, idéal pour la visualisation 2D

On encode d'abord les variables catégorielles pour pouvoir les utiliser.

In [ ]:
# Préparation : encoder les colonnes catégorielles
df_encoded = df.copy()

cat_cols = ['experience_level', 'employment_type', 'job_title',
            'employee_residence', 'company_location', 'company_size']

le = LabelEncoder()
for col in cat_cols:
    if col in df_encoded.columns:
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# Sélectionner les colonnes numériques pour la réduction
num_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
# Exclure les colonnes de salaire brut et normalisé pour éviter la redondance
features = [c for c in num_cols if c not in ['salary', 'salary_in_usd', 'salary_normalized']]

X = df_encoded[features].fillna(0)
print(f'Features utilisées pour la réduction ({len(features)}) : {features}')

In [ ]:
# ── PCA ──────────────────────────────────────────────────────────────────────
# Réduire à 2 composantes principales pour la visualisation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

print(f'Variance expliquée par chaque composante : {pca.explained_variance_ratio_.round(4)}')
print(f'Variance totale expliquée : {pca.explained_variance_ratio_.sum():.2%}')

# Créer un DataFrame pour la visualisation
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['experience_level'] = df['experience_level'].values

# Mapping des codes vers des labels lisibles
exp_labels = {'EN': 'Junior', 'MI': 'Intermédiaire', 'SE': 'Senior', 'EX': 'Expert'}
pca_df['experience_level'] = pca_df['experience_level'].map(exp_labels).fillna(pca_df['experience_level'])

# Visualisation PCA
plt.figure(figsize=(9, 6))
colors = {'Junior': '#4e9af1', 'Intermédiaire': '#f1a94e', 'Senior': '#4ef169', 'Expert': '#f14e4e'}
for level, group in pca_df.groupby('experience_level'):
    plt.scatter(group['PC1'], group['PC2'], label=level,
                alpha=0.6, s=40, color=colors.get(level, 'gray'))

plt.title('PCA — Réduction à 2 composantes principales\n(coloré par niveau d\'expérience)', fontsize=13)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} de variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} de variance)')
plt.legend(title="Niveau d'expérience")
plt.tight_layout()
plt.show()

In [ ]:
# Scree plot — variance expliquée cumulée
pca_full = PCA(random_state=42)
pca_full.fit(X)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumvar)+1), cumvar, marker='o', color='steelblue')
plt.axhline(y=0.95, color='red', linestyle='--', label='Seuil 95%')
plt.title('Variance expliquée cumulée — PCA')
plt.xlabel('Nombre de composantes')
plt.ylabel('Variance expliquée cumulée')
plt.legend()
plt.tight_layout()
plt.show()

n_95 = np.argmax(cumvar >= 0.95) + 1
print(f'📌 {n_95} composantes suffisent pour expliquer 95% de la variance.')

In [ ]:
# ── t-SNE ─────────────────────────────────────────────────────────────────────
# t-SNE est plus lent mais révèle mieux les clusters
# On limite à 500 lignes pour la rapidité (optionnel)
sample_idx = df_encoded.sample(n=min(500, len(df_encoded)), random_state=42).index
X_sample = X.loc[sample_idx]
exp_sample = df['experience_level'].loc[sample_idx].map(exp_labels).fillna(df['experience_level'].loc[sample_idx])

print('Calcul du t-SNE en cours (peut prendre quelques secondes)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_sample)

tsne_df = pd.DataFrame(X_tsne, columns=['Dim1', 'Dim2'])
tsne_df['experience_level'] = exp_sample.values

# Visualisation t-SNE
plt.figure(figsize=(9, 6))
for level, group in tsne_df.groupby('experience_level'):
    plt.scatter(group['Dim1'], group['Dim2'], label=level,
                alpha=0.7, s=40, color=colors.get(level, 'gray'))

plt.title('t-SNE — Réduction à 2 dimensions\n(coloré par niveau d\'expérience)', fontsize=13)
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.legend(title="Niveau d'expérience")
plt.tight_layout()
plt.show()
print('✅ t-SNE calculé.')

---
## 🌟 Tâche 3 : Agrégation par Niveau d'Expérience

Calculer le **salaire moyen** et **médian** par niveau d'expérience.

| Code | Signification |
|------|---------------|
| EN   | Entry-level / Junior |
| MI   | Mid-level / Intermédiaire |
| SE   | Senior |
| EX   | Executive / Expert |

In [ ]:
# Agrégation par niveau d'expérience
agg = df.groupby('experience_level')['salary_in_usd'].agg(
    Effectif='count',
    Salaire_Moyen='mean',
    Salaire_Médian='median',
    Salaire_Min='min',
    Salaire_Max='max',
    Écart_Type='std'
).round(0)

# Ajouter les labels lisibles
agg.index = agg.index.map(exp_labels)
agg = agg.sort_values('Salaire_Moyen')

print('=== Salaires par niveau d\'expérience (en USD) ===')
display(agg)

In [ ]:
# Graphique barres : salaire moyen vs médian
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(agg))
width = 0.35

bars1 = ax.bar(x - width/2, agg['Salaire_Moyen'], width, label='Salaire Moyen',
               color='steelblue', edgecolor='white')
bars2 = ax.bar(x + width/2, agg['Salaire_Médian'], width, label='Salaire Médian',
               color='teal', edgecolor='white')

# Annotations
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{bar.get_height():,.0f}$", ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{bar.get_height():,.0f}$", ha='center', va='bottom', fontsize=9)

ax.set_title("Salaire Moyen vs Médian par Niveau d'Expérience", fontsize=13)
ax.set_xlabel("Niveau d'expérience")
ax.set_ylabel('Salaire (USD)')
ax.set_xticks(x)
ax.set_xticklabels(agg.index)
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}$'))
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot détaillé par niveau d'expérience
df_plot = df.copy()
df_plot['experience_level'] = df_plot['experience_level'].map(exp_labels).fillna(df_plot['experience_level'])

order = ['Junior', 'Intermédiaire', 'Senior', 'Expert']
palette = {'Junior': '#4e9af1', 'Intermédiaire': '#f1a94e', 'Senior': '#4ef169', 'Expert': '#f14e4e'}

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_plot,
    x='experience_level',
    y='salary_in_usd',
    order=[o for o in order if o in df_plot['experience_level'].unique()],
    palette=palette
)
plt.title("Distribution des Salaires par Niveau d'Expérience", fontsize=13)
plt.xlabel("Niveau d'expérience")
plt.ylabel('Salaire (USD)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}$'))
plt.tight_layout()
plt.show()

In [ ]:
# Évolution du salaire moyen avec l'expérience
plt.figure(figsize=(8, 4))
plt.plot(agg.index, agg['Salaire_Moyen'], marker='o', linewidth=2,
         color='steelblue', label='Salaire moyen')
plt.plot(agg.index, agg['Salaire_Médian'], marker='s', linewidth=2,
         color='teal', linestyle='--', label='Salaire médian')
plt.fill_between(agg.index,
                 agg['Salaire_Moyen'] - agg['Écart_Type'],
                 agg['Salaire_Moyen'] + agg['Écart_Type'],
                 alpha=0.15, color='steelblue', label='± 1 écart-type')
plt.title("Évolution du Salaire avec le Niveau d'Expérience", fontsize=13)
plt.xlabel("Niveau d'expérience")
plt.ylabel('Salaire (USD)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}$'))
plt.legend()
plt.tight_layout()
plt.show()

---
## ✅ Résumé des Conclusions

In [ ]:
print('=' * 60)
print('RÉSUMÉ DES CONCLUSIONS')
print('=' * 60)

print('\n📊 TÂCHE 1 — Normalisation Min-Max')
print(f"  Plage originale : {df['salary_in_usd'].min():,.0f}$ → {df['salary_in_usd'].max():,.0f}$")
print(f"  Plage normalisée : {df['salary_normalized'].min():.2f} → {df['salary_normalized'].max():.2f}")
print('  → La forme de la distribution est conservée, seule l\'échelle change.')

print('\n🔬 TÂCHE 2 — Réduction de dimensionnalité')
print(f"  PCA : 2 composantes expliquent {pca.explained_variance_ratio_.sum():.1%} de la variance.")
print(f"  PCA : {n_95} composantes suffisent pour atteindre 95% de variance expliquée.")
print('  t-SNE : révèle des regroupements par niveau d\'expérience.')

print('\n💰 TÂCHE 3 — Agrégation par niveau d\'expérience')
for level in agg.index:
    print(f"  {level:<20} | Moyenne : {agg.loc[level, 'Salaire_Moyen']:>10,.0f}$ | Médiane : {agg.loc[level, 'Salaire_Médian']:>10,.0f}$")

print('\n📌 Observations clés :')
print('  - Le salaire augmente clairement avec le niveau d\'expérience.')
print('  - La moyenne > médiane → distribution asymétrique à droite (quelques très hauts salaires).')
print('  - Les experts ont la plus grande dispersion salariale (écart-type élevé).')

In [ ]:
# Sauvegarder le dataset final enrichi
df.to_csv('ds_salaries_processed.csv', index=False)
print('✅ Dataset enrichi sauvegardé : ds_salaries_processed.csv')